# Robust DPO — Results Analysis
Reproduces Tables 2 & 3 and the key figures from Hooda & Kumar (2025).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path('.').resolve()))
from src.utils import collect_all_results

RESULTS_DIR = 'results'   # change if needed
METHOD_ORDER = ['vanilla', 'cdpo', 'rdpo', 'ropo']
METHOD_LABELS = {
    'vanilla': 'Vanilla DPO',
    'cdpo':    'cDPO',
    'rdpo':    'rDPO',
    'ropo':    'ROPO',
}
METHOD_COLORS = {
    'vanilla': '#e15759',
    'cdpo':    '#f28e2b',
    'rdpo':    '#4e79a7',
    'ropo':    '#59a14f',
}
METHOD_MARKERS = {'vanilla': 'o', 'cdpo': 's', 'rdpo': '^', 'ropo': 'D'}

sns.set_theme(style='whitegrid', font_scale=1.15)
print('Setup complete.')

## Load results

In [ ]:
rows = collect_all_results(RESULTS_DIR)

# Also merge in judge results if available
for row in rows:
    judge_path = Path(RESULTS_DIR) / f"{row['method']}_noise{int(row['flip_prob']*100):02d}" / 'judge_results.json'
    if judge_path.exists():
        with open(judge_path) as f:
            j = json.load(f)
        row['win_rate'] = j.get('win_rate', None)

df = pd.DataFrame(rows)
df['noise_pct'] = (df['flip_prob'] * 100).astype(int)
df = df.sort_values(['noise_pct', 'method'])

# Normalise method order
df['method'] = pd.Categorical(df['method'], categories=METHOD_ORDER, ordered=True)
df = df.sort_values(['noise_pct', 'method'])
print(f'Loaded {len(df)} results rows.')
df.head(10)

## Table 2 — Full results

In [ ]:
cols = ['noise_pct', 'method', 'avg_train_loss', 'eval_margin', 'eval_accuracy', 'win_rate']
present = [c for c in cols if c in df.columns]

display_df = df[present].copy()
display_df['method'] = display_df['method'].map(METHOD_LABELS)
display_df.columns = ['Noise (%)', 'Method', 'Train Loss', 'Eval Margin', 'Eval Acc.', 'Win Rate (%)'][:len(present)]

# Bold best per noise level (by Eval Margin)
styled = (
    display_df.style
    .format({'Train Loss': '{:.3f}', 'Eval Margin': '{:.3f}',
             'Eval Acc.': '{:.3f}', 'Win Rate (%)': '{:.1f}'}, na_rep='—')
    .highlight_max(subset=['Eval Margin', 'Eval Acc.', 'Win Rate (%)'], color='lightgreen')
    .highlight_min(subset=['Train Loss'], color='lightgreen')
    .set_caption('Table 2: Full results across noise levels and methods.')
)
styled

## Table 3 — Robustness delta vs vanilla

In [ ]:
vanilla_margins = df[df['method'] == 'vanilla'].set_index('noise_pct')['eval_margin']

delta_rows = []
for _, row in df[df['method'] != 'vanilla'].iterrows():
    delta_rows.append({
        'Noise (%)': row['noise_pct'],
        'Method':    METHOD_LABELS[row['method']],
        'Δ Eval Margin': row['eval_margin'] - vanilla_margins.get(row['noise_pct'], float('nan')),
    })

delta_df = pd.DataFrame(delta_rows)
pivot = delta_df.pivot(index='Noise (%)', columns='Method', values='Δ Eval Margin')

pivot.style \
    .format('{:+.3f}') \
    .background_gradient(cmap='RdYlGn', axis=None) \
    .set_caption('Table 3: Robustness delta (Δ Eval Margin vs vanilla DPO). Positive = better than vanilla.')

## Figure 1 — Eval Margin vs Noise

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for method in METHOD_ORDER:
    sub = df[df['method'] == method].sort_values('noise_pct')
    ax.plot(
        sub['noise_pct'], sub['eval_margin'],
        marker=METHOD_MARKERS[method],
        color=METHOD_COLORS[method],
        label=METHOD_LABELS[method],
        linewidth=2, markersize=7,
    )

ax.set_xlabel('Label Noise Rate (%)')
ax.set_ylabel('Eval Margin (↑ better)')
ax.set_title('Preference Margin vs Noise Level')
ax.legend(framealpha=0.9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/fig_margin_vs_noise.pdf', dpi=150)
plt.show()

## Figure 2 — GPT-4o Win Rate vs Noise

In [ ]:
if 'win_rate' in df.columns and df['win_rate'].notna().any():
    fig, ax = plt.subplots(figsize=(7, 4.5))

    for method in METHOD_ORDER:
        sub = df[(df['method'] == method) & df['win_rate'].notna()].sort_values('noise_pct')
        if sub.empty:
            continue
        ax.plot(
            sub['noise_pct'], sub['win_rate'],
            marker=METHOD_MARKERS[method],
            color=METHOD_COLORS[method],
            label=METHOD_LABELS[method],
            linewidth=2, markersize=7,
        )

    ax.axhline(50, color='grey', linestyle='--', linewidth=1, label='Parity (50%)')
    ax.set_xlabel('Label Noise Rate (%)')
    ax.set_ylabel('GPT-4o Win Rate (%, ↑ better)')
    ax.set_title('LLM-as-a-Judge Win Rate vs Noise Level')
    ax.legend(framealpha=0.9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    fig.tight_layout()
    fig.savefig(f'{RESULTS_DIR}/fig_winrate_vs_noise.pdf', dpi=150)
    plt.show()
else:
    print('No win_rate data found. Run scripts/run_judge.py first.')

## Figure 3 — Preference Accuracy vs Noise

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

for method in METHOD_ORDER:
    sub = df[df['method'] == method].sort_values('noise_pct')
    ax.plot(
        sub['noise_pct'], sub['eval_accuracy'] * 100,
        marker=METHOD_MARKERS[method],
        color=METHOD_COLORS[method],
        label=METHOD_LABELS[method],
        linewidth=2, markersize=7,
    )

ax.axhline(50, color='grey', linestyle='--', linewidth=1, label='Random (50%)')
ax.set_xlabel('Label Noise Rate (%)')
ax.set_ylabel('Preference Accuracy (%, ↑ better)')
ax.set_title('Preference Accuracy vs Noise Level')
ax.legend(framealpha=0.9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
fig.tight_layout()
fig.savefig(f'{RESULTS_DIR}/fig_accuracy_vs_noise.pdf', dpi=150)
plt.show()

## Figure 4 — Training Loss Curves (per method, stacked)

In [ ]:
# Load per-step loss arrays if saved
loss_data = {}
for method in METHOD_ORDER:
    for flip_prob in df['flip_prob'].unique():
        noise_pct = int(flip_prob * 100)
        metrics_path = Path(RESULTS_DIR) / f'{method}_noise{noise_pct:02d}' / 'metrics.json'
        if metrics_path.exists():
            with open(metrics_path) as f:
                m = json.load(f)
            if 'losses' in m:
                loss_data[(method, noise_pct)] = m['losses']

if loss_data:
    noise_levels = sorted(df['noise_pct'].unique())
    fig, axes = plt.subplots(1, len(noise_levels), figsize=(4 * len(noise_levels), 4), sharey=True)

    for ax, noise_pct in zip(axes, noise_levels):
        for method in METHOD_ORDER:
            losses = loss_data.get((method, noise_pct))
            if losses:
                ax.plot(losses, color=METHOD_COLORS[method], label=METHOD_LABELS[method], linewidth=1.2)
        ax.set_title(f'Noise = {noise_pct}%')
        ax.set_xlabel('Gradient Step')
        if ax == axes[0]:
            ax.set_ylabel('Training Loss')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', framealpha=0.9)
    fig.suptitle('Training Loss Curves', y=1.02)
    fig.tight_layout()
    fig.savefig(f'{RESULTS_DIR}/fig_loss_curves.pdf', dpi=150)
    plt.show()
else:
    print('Per-step loss arrays not found in metrics.json (run with save enabled).')

## Robustness Tax at 0% Noise

In [ ]:
zero_noise = df[df['noise_pct'] == 0].copy()
vanilla_margin_0 = zero_noise[zero_noise['method'] == 'vanilla']['eval_margin'].values[0]

print('Robustness tax at 0% noise (delta vs vanilla):')
for _, row in zero_noise[zero_noise['method'] != 'vanilla'].iterrows():
    delta = row['eval_margin'] - vanilla_margin_0
    print(f"  {METHOD_LABELS[row['method']]:<15}: Δ = {delta:+.4f}")
print(f"\nAll robust methods within ±0.009 of vanilla at 0% noise (paper claim).")